<a href="https://colab.research.google.com/github/en970/gausscapture/blob/main/notebooks/GaussCapture_Colab_4D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GaussCapture · Colab 4D trainer

Trains a **deformable** Gaussian scene from a `dataset4d.zip` produced by

    gausscapture prep4d <project>
    gausscapture colab4d <project>

**Runtime → Change runtime type → GPU.** An L4 or A100 is comfortable; a T4
works at `--cap-max 200000`. Budget 30–60 minutes.

---

### What this produces, and what it does not

The capture is a phone on a fixed support pointed at your own face. During the
`hold` phase the camera does not move, so those frames contain **no parallax**:
they cannot, by themselves, tell anyone where anything is in depth. The geometry
comes from the `arc` — a hand-held sweep recorded while the subject held still,
which makes the head part of the rigid scene and lets ordinary triangulation
recover it.

So the result is a **bullet-time scene bounded by the cone the arc actually
swept**. That cone was measured from the solved arc cameras during `prep4d`, it
travels inside `scene4d_init.npz`, and the viewer clamps to it. Nothing here
produces a free-viewpoint scene, and nothing here should be described as one.

### Why this notebook installs our trainer rather than someone else's

There is no permissively licensed off-the-shelf 4D Gaussian trainer to drive.
The published fixed-camera work depends on the reference implementation's
nearest-neighbour helper and its differential rasterizer, both licensed for
**non-commercial research only**, which GaussCapture's MIT licence cannot
absorb. (Their package names are deliberately not written here: the licence
gate in CI fails on any file that contains them, which is what stops one being
installed by a notebook cell nobody reread.) So the deformation field, the
scaffold, the losses and the schedule are implemented in this repository, in
plain PyTorch, and the only rasterizer imported is
[gsplat](https://github.com/nerfstudio-project/gsplat) — Apache-2.0. Nothing
non-commercial is cloned, installed, or linked at any point in this notebook.

## 1 · GPU and Drive

Output goes to Google Drive, not `/content`. Colab recycles idle runtimes and
wipes `/content` when it does, and a 45-minute run is exactly long enough to
lose that way.

**Put `dataset4d.zip` in the root of your Drive first** (drag it onto
drive.google.com). Colab's upload widget is avoided deliberately: it breaks when
the cell output is restored from an earlier browser session.

In [ ]:
import pathlib, shutil, zipfile

import torch

assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU'
NAME = torch.cuda.get_device_name(0)
MAJOR, MINOR = torch.cuda.get_device_capability(0)
print(f'{NAME} · sm_{MAJOR}{MINOR} · torch {torch.__version__} · cuda {torch.version.cuda}')

from google.colab import drive
drive.mount('/content/drive')

DRIVE = pathlib.Path('/content/drive/MyDrive')
OUT = DRIVE / 'gausscapture_4d'      # survives a recycled runtime
OUT.mkdir(exist_ok=True)

found = sorted(DRIVE.glob('dataset4d*.zip'))
assert found, 'Put dataset4d.zip in the root of your Drive, then rerun.'
DATASET = found[0]
print(f'dataset: {DATASET.name} · {DATASET.stat().st_size / 1e6:.1f} MB')

# Inspect it before spending a GPU hour on it: the init file is the trainer's
# whole input, and an empty or half-written one is cheaper to find now.
with zipfile.ZipFile(DATASET) as z:
    names = z.namelist()
images = [n for n in names if n.startswith('images/') and not n.endswith('/')]
assert 'scene4d_init.npz' in names, 'No scene4d_init.npz — repackage with `gausscapture colab4d`.'
print(f'{len(images)} hold frames · {len(names)} members')

## 2 · The permissive GPU stack

Three things here are not optional, and each of them fails in a way that points
somewhere else:

* **torch is pinned to a cu128 build.** gsplat's newest verified CUDA is 12.8.
  Colab's default torch has tracked CUDA 13.0, and gsplat's kernels do not
  compile against it — the error surfaces from `nvcc` deep inside a JIT build
  and reads like a missing header rather than a version mismatch.
* **gsplat comes from git main, not PyPI.** The released 1.5.3 has a dead
  opacity reset: `if step % self.reset_every == 0 & step > 0` parses as a
  chained comparison that is always false. We train with MCMC, so it does not
  bite us directly, but a release whose scheduling is silently inert is not one
  to reason about.
* **`TORCH_CUDA_ARCH_LIST` names one architecture — this GPU's.** Left unset,
  the JIT compiles for every architecture it can think of, which turns a
  two-minute build into a twenty-minute one.

The build directory is cached on Drive, so a recycled runtime recompiles
nothing.

In [ ]:
import os, subprocess, sys

os.environ['TORCH_CUDA_ARCH_LIST'] = f'{MAJOR}.{MINOR}'
BUILD_CACHE = OUT / 'torch_extensions'
BUILD_CACHE.mkdir(exist_ok=True)
os.environ['TORCH_EXTENSIONS_DIR'] = str(BUILD_CACHE)


def pip(*args: str) -> None:
    done = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args],
                          capture_output=True, text=True)
    label = args[-1].split('@')[0].split('/')[-1]
    print(f"  {'OK  ' if done.returncode == 0 else 'FAIL'} {label}")
    if done.returncode != 0:
        print(done.stderr.strip()[-800:])   # the tail is where the cause is


# Installed one at a time. `pip install -r` is all-or-nothing: one package that
# fails to build takes the list with it, and the first symptom is an unrelated
# ModuleNotFoundError much later.
if not torch.version.cuda.startswith('12.8'):
    pip('--index-url', 'https://download.pytorch.org/whl/cu128', 'torch', 'torchvision')
    print('torch replaced — RUNTIME > RESTART SESSION, then rerun cells 1 and 2.')
else:
    pip('git+https://github.com/nerfstudio-project/gsplat.git')
    pip('scipy', 'tqdm')

## 3 · GaussCapture itself

Installed with `--no-deps`. Its dependencies are numpy and OpenCV, both of which
Colab already has at versions that work; letting pip resolve them invites it to
reinstall numpy, which breaks the torch build that was just pinned.

In [ ]:
pip('--no-deps', 'git+https://github.com/en970/gausscapture.git@main')

check = subprocess.run(
    [sys.executable, '-c',
     'import gsplat, numpy, torch\n'
     'from gausscapture.recon.fit4d import train_4d\n'
     'print("imports OK · gsplat", gsplat.__version__, "· numpy", numpy.__version__)'],
    capture_output=True, text=True)
print(check.stdout)
if check.returncode != 0:
    print(check.stderr)
    raise SystemExit('fix the import above before training')

## 4 · Train

Two stages, and the split is the whole method:

1. **Coarse.** Deformation is forced to identity, so every frame is fitted by
   one static scene. MCMC densifies up to a hard `cap_max`, which is also the
   `.g4d` payload budget. Densifying here rather than later matters: gradient-
   driven splitting on *deformed* geometry splits gaussians because they are
   moving, not because their canonical geometry is inadequate.
2. **Fine.** The SE(3) scaffold turns on, densification stops, and the MCMC
   position noise is driven to zero — it mutates canonical means in place, and
   the deformation field is indexed by canonical position.

Output streams live and lands in Drive, so neither a long silence nor a recycled
runtime costs you the run. `--resume` is what makes that literally true: a
`scene4d.ckpt` is written to Drive every 1000 steps, and rerunning this cell
continues from the last one rather than starting the 15,000 iterations again.

In [ ]:
SCENE = OUT / 'scene4d.npz'
CAP_MAX = 400_000      # 200_000 on a T4
COARSE_ITERS = 3_000
ITERS = 12_000

command = [sys.executable, '-u', '-m', 'gausscapture.cli', 'train4d', str(DATASET),
           '--out', str(SCENE),
           '--coarse-iters', str(COARSE_ITERS),
           '--iters', str(ITERS),
           '--cap-max', str(CAP_MAX),
           '--device', 'cuda',
           # Writes scene4d.ckpt beside scene4d.npz in Drive every 1000 steps,
           # and picks up from it if this cell is rerun after a recycled runtime.
           '--resume']
print('$', ' '.join(command), end='\n\n')

# -u plus a line-buffered Popen: progress appears as it happens rather than in
# one block at the end, so a long run does not look like a hang. GaussCapture
# writes progress to stderr and data to stdout, and both are merged here.
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
print('\nDONE' if process.wait() == 0 else '\nFAILED')

## 5 · Collect

`scene4d.npz` is already in Drive and stays there. Everything after this point
runs on your own machine — the export and the viewer need no GPU:

    gausscapture colab4d <project> --collect ~/Downloads/scene4d.npz
    gausscapture export4d <project> --out scene.g4d
    gausscapture viewer4d <project> --serve

One number to read carefully. If the training log printed a hold-out PSNR, it
was measured by withholding every eighth `hold` frame — the same camera, a
different instant. It measures **motion interpolation**. There is no held-out
viewpoint anywhere in this design, so that figure is not comparable to a masked
mPSNR from DyCheck or N3DV, and putting it in the same table as one would be a
claim this capture cannot support.

In [ ]:
assert SCENE.exists(), 'No scene4d.npz — read the training log above.'
import numpy as np

with np.load(SCENE, allow_pickle=False) as data:
    print(f'{SCENE}  {SCENE.stat().st_size / 1e6:.1f} MB')
    for key in data.files:
        print(f'  {key:<20} {data[key].shape} {data[key].dtype}')

from google.colab import files
files.download(str(SCENE))